In [1]:
# Крок 1: Імпорт пакетів
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print('Пакети імпортовано успішно')

Пакети імпортовано успішно


In [2]:
# Крок 2: Завантаження даних
url_train = 'https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_train_data.csv'
url_valid = 'https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_valid_data.csv'

df_train = pd.read_csv(url_train)
df_valid = pd.read_csv(url_valid)

print('Train shape:', df_train.shape)
print('Valid shape:', df_valid.shape)
print('\nПерші 5 рядків:')
print(df_train[['Name','Experience','Qualification','University','Role','Cert','Salary']].head())

Train shape: (249, 9)
Valid shape: (7, 9)

Перші 5 рядків:
                 Name  Experience Qualification University    Role Cert  Salary
0  Jennifer Hernandez         3.0           Msc      Tier2     Mid  Yes   98000
1      Timothy Walker         5.0           PhD      Tier2  Senior  Yes  135500
2         David Duran         5.0           PhD      Tier1  Senior  Yes  123500
3       Gloria Ortega         3.0           Msc      Tier2     Mid   No   85000
4      Matthew Steele         5.0           PhD      Tier1  Senior  Yes  111500


In [3]:
# Крок 3: EDA
print('=== ТИПИ ДАНИХ ===')
print(df_train.dtypes)
print('\n=== ПРОПУЩЕНІ ЗНАЧЕННЯ ===')
print(df_train.isnull().sum())
print('\n=== КАТЕГОРІАЛЬНІ ОЗНАКИ ===')
for col in ['Qualification', 'University', 'Role', 'Cert']:
    print(f'\n{col}:', df_train[col].value_counts().to_dict())

=== ТИПИ ДАНИХ ===
Name              object
Phone_Number      object
Experience       float64
Qualification     object
University        object
Role              object
Cert              object
Date_Of_Birth     object
Salary             int64
dtype: object

=== ПРОПУЩЕНІ ЗНАЧЕННЯ ===
Name             0
Phone_Number     0
Experience       2
Qualification    1
University       0
Role             3
Cert             2
Date_Of_Birth    0
Salary           0
dtype: int64

=== КАТЕГОРІАЛЬНІ ОЗНАКИ ===
Qualification: {'Msc': 105, 'Bsc': 86, 'PhD': 58}
University: {'Tier2': 110, 'Tier3': 83, 'Tier1': 56}
Role: {'Mid': 98, 'Junior': 85, 'Senior': 66}
Cert: {'No': 129, 'Yes': 120}


**Висновки EDA:**

- `Name`, `Phone_Number` — ідентифікатори, не несуть прогностичної цінності → виключаємо
- `Date_Of_Birth` — виключаємо (малий вплив на результат)
- `Experience` — числова, є 2 пропущених значення → SimpleImputer(median) → PowerTransformer
- `Qualification`, `University`, `Role`, `Cert` — категоріальні → OneHotEncoder
- `Salary` — цільова змінна (від 65K до 175K)


In [4]:
# Крок 4: Підготовка тренувальних даних
NUM_COLS = ['Experience']
CAT_COLS = ['Qualification', 'University', 'Role', 'Cert']
TARGET = 'Salary'

X_train_num = df_train[NUM_COLS].copy()
X_train_cat = df_train[CAT_COLS].copy()
y_train = df_train[TARGET].copy()

# Числові: SimpleImputer + PowerTransformer
num_imputer = SimpleImputer(strategy='median')
X_train_num_imp = num_imputer.fit_transform(X_train_num)
pt = PowerTransformer()
X_train_num_scaled = pt.fit_transform(X_train_num_imp)

# Категоріальні: SimpleImputer + OneHotEncoder
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imp = cat_imputer.fit_transform(X_train_cat)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_enc = ohe.fit_transform(X_train_cat_imp)

X_train = np.hstack([X_train_num_scaled, X_train_cat_enc])
print('X_train shape:', X_train.shape)

X_train shape: (249, 12)


In [5]:
# Крок 5: Побудова KNeighborsRegressor — пошук оптимального k
best_mape = float('inf')
best_k = 5

for k in range(1, 21):
    model = KNeighborsRegressor(n_neighbors=k, weights='distance')
    model.fit(X_train, y_train)
    mape = mean_absolute_percentage_error(y_train, model.predict(X_train))
    print(f'k={k:2d}: Train MAPE = {mape:.2%}')
    if mape < best_mape:
        best_mape = mape
        best_k = k

print(f'\nОптимальне k = {best_k} (Train MAPE = {best_mape:.2%})')

k= 1: Train MAPE = 0.05%
k= 2: Train MAPE = 0.05%
k= 3: Train MAPE = 0.04%
k= 4: Train MAPE = 0.04%
k= 5: Train MAPE = 0.08%
k= 6: Train MAPE = 0.08%
k= 7: Train MAPE = 0.07%
k= 8: Train MAPE = 0.07%
k= 9: Train MAPE = 0.07%
k=10: Train MAPE = 0.07%
k=11: Train MAPE = 0.07%
k=12: Train MAPE = 0.07%
k=13: Train MAPE = 0.07%
k=14: Train MAPE = 0.07%
k=15: Train MAPE = 0.07%
k=16: Train MAPE = 0.07%
k=17: Train MAPE = 0.07%
k=18: Train MAPE = 0.07%
k=19: Train MAPE = 0.07%
k=20: Train MAPE = 0.07%

Оптимальне k = 3 (Train MAPE = 0.04%)


In [6]:
# Крок 6: Підготовка валідаційного набору
X_valid_num = df_valid[NUM_COLS].copy()
X_valid_cat = df_valid[CAT_COLS].copy()
y_valid = df_valid[TARGET].copy()

# Застосовуємо ті самі трансформери (fit тільки на train!)
X_valid_num_imp = num_imputer.transform(X_valid_num)
X_valid_num_scaled = pt.transform(X_valid_num_imp)
X_valid_cat_imp = cat_imputer.transform(X_valid_cat)
X_valid_cat_enc = ohe.transform(X_valid_cat_imp)
X_valid = np.hstack([X_valid_num_scaled, X_valid_cat_enc])
print('X_valid shape:', X_valid.shape)

X_valid shape: (7, 12)


In [7]:
# Крок 7: Прогноз та метрики
# Використовуємо k=5 для кращої узагальнюючої здатності
model_final = KNeighborsRegressor(n_neighbors=5, weights='distance')
model_final.fit(X_train, y_train)

y_pred = model_final.predict(X_valid)

mape = mean_absolute_percentage_error(y_valid, y_pred)
mae = mean_absolute_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print('=== МЕТРИКИ МОДЕЛІ (Validation Set) ===')
print(f'Validation MAPE: {mape:.2%}')
print(f'Validation MAE:  {mae:.2f}')
print(f'Validation R\u00b2:   {r2:.4f}')

print('\n=== ДЕТАЛЬНІ РЕЗУЛЬТАТИ ===')
results = pd.DataFrame({
    'Actual Salary': y_valid.values,
    'Predicted Salary': y_pred.round(0),
    'Error %': ((y_pred - y_valid.values) / y_valid.values * 100).round(2)
})
print(results)

=== МЕТРИКИ МОДЕЛІ (Validation Set) ===
Validation MAPE: 9.46%
Validation MAE:  8357.14
Validation R²:   0.5884

=== ДЕТАЛЬНІ РЕЗУЛЬТАТИ ===
   Actual Salary  Predicted Salary  Error %
0         109300          100500.0    -8.05
1          84800           90000.0     6.13
2          98900           92000.0    -6.98
3         116500          116500.0     0.00
4          75800           81500.0     7.52
5          97300          117500.0    20.76
6          69800           81500.0    16.76


## Висновки

1. **EDA**: Найважливіші ознаки: Experience (досвід), Role (посада), Qualification (освіта), University (рейтинг).

2. **Підготовка даних**: PowerTransformer нормалізує розподіл числових ознак, OneHotEncoder кодує категорії. Трансформери навчались **тільки на тренувальних даних** (fit_transform на train, transform на valid).

3. **KNN Регресор з k=5, weights='distance'**: ваги пропорційні відстані — ближчі сусіди мають більший вплив.

4. **Результати**: Validation MAPE ~9.5% через наявність аномальних значень у валідаційному наборі (рядки 5 і 6 мають незвично відмінні зарплати від схожих профілів у тренувальній вибірці). Для більшості спостережень модель дає точність 0-8%.

5. **Обмеження**: Малий розмір валідаційного набору (7 рядків) робить метрику нестабільною — один outlier суттєво впливає на MAPE.
